# 05 — Proposta di Separazione delle Leggi Ibride

Questo notebook produce `splits.json`, caricato staticamente dall'HTML per il toggle Prima/Dopo.

| Fase | Input | Operazione | Output |
|---|---|---|---|
| **A** | `nodes_hybridity.csv`, `nodes_lamfalussy.csv` | Selezione candidati e partizione deterministica | `split_candidates` dict |
| **B** | Testo completo articoli | LLM: valida partizione e propone titoli | `split_validated` dict |
| **C** | `segments_lamfalussy.csv` + testi | Attribuzione archi a livello articolo | `article_edges` dict |
| **D** | Archi con target splittato | LLM: risolve ambiguità target | `edges_resolved` list |
| **E** | Tutto | Calcolo H globale prima/dopo + serializzazione | `splits.json` |

## 0. Configurazione

**Modifica solo questa cella.**

In [ ]:
MATERIA_NAME = "appalti_it"

# ── Modello ────────────────────────────────────────────────────────────────────
LLM_MODEL         = "gpt-4.1-mini"
LLM_MAX_TOKENS    = 3000
LLM_DELAY_SECONDS = 0.3
LLM_MAX_RETRIES   = 3
LLM_RETRY_DELAY   = 5.0
MAX_WORKERS       = 4

# ── Soglie split ───────────────────────────────────────────────────────────────
HYBRIDITY_THRESHOLD   = 0.35   # H minima per proporre uno split
MIN_ARTICLES_FOR_SPLIT = 8     # atti con meno articoli non vengono splittati
MIN_BLOCK_SIZE         = 4     # blocchi con meno articoli vengono assorbiti
MIN_H_IMPROVEMENT      = 0.30  # miglioramento minimo di H per rendere lo split utile

# ── Checkpoint ────────────────────────────────────────────────────────────────
CHECKPOINT_EVERY = 5

# ── Visualizzazione HTML ───────────────────────────────────────────────────────
# True  = appalti_it e domini con HTML; False = domini senza HTML (es. fdi_screening)
ENABLE_HTML_OUTPUT = True

## 1. Import e Percorsi

In [ ]:
import os, re, json, math, time
import numpy as np
import pandas as pd
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor, as_completed
from threading import Lock
from dotenv import load_dotenv
load_dotenv(dotenv_path=r'C:\Users\claud\Documents\GitHub\eu-law-network-viz\.env')

from openai import OpenAI
import openai

client = OpenAI()

# ── Override via variabile d'ambiente (per esecuzione programmatica) ──────────
_materia_override = os.environ.get('MATERIA_OVERRIDE', '')
if _materia_override:
    MATERIA_NAME = _materia_override
_html_override = os.environ.get('ENABLE_HTML_OVERRIDE', '')
if _html_override:
    ENABLE_HTML_OUTPUT = _html_override.lower() in ('true', '1', 'yes')

# ── Percorsi ──────────────────────────────────────────────────────────────────
output_path = os.path.join('..', 'data', 'output', MATERIA_NAME)

NODES_HYBRIDITY_FILE   = os.path.join(output_path, 'nodes_hybridity.csv')
NODES_LAMFALUSSY_FILE  = os.path.join(output_path, 'nodes_lamfalussy.csv')
SEGMENTS_LAMF_FILE     = os.path.join(output_path, 'segments_lamfalussy.csv')
NODES_DIFFUSION_FILE   = os.path.join(output_path, 'nodes_diffusion.csv')

# Auto-detect file nodi: prova prima nodes_texts_it.csv (appalti), poi nodes_texts.csv
_it_path  = os.path.join(output_path, 'nodes_texts_it.csv')
_all_path = os.path.join(output_path, 'nodes_texts.csv')
NODES_TEXTS_IT_FILE = _it_path if os.path.exists(_it_path) else _all_path

# Auto-detect file nodi EU (cerca nodes_texts_eu_*.csv, altrimenti stringa vuota)
import glob as _glob_nb05
_eu_candidates = _glob_nb05.glob(os.path.join(output_path, 'nodes_texts_eu_*.csv'))
NODES_TEXTS_EU_FILE = _eu_candidates[0] if _eu_candidates else ''

EDGES_IT_FILE          = os.path.join(output_path, 'edges_it_internal.csv')
EDGES_EU_IT_FILE       = os.path.join(output_path, 'edges_eu_it.csv')
HEATMAPS_FILE          = os.path.join(output_path, 'heatmaps.json')
SPLITS_CKPT_FILE       = os.path.join(output_path, 'splits_checkpoint.json')
SPLITS_JSON_FILE       = os.path.join(output_path, 'splits.json')

# File HTML (None se ENABLE_HTML_OUTPUT=False)
HTML_FILE = os.path.join('..', f'{MATERIA_NAME}_network.html') if ENABLE_HTML_OUTPUT else None

# Se HTML_FILE e' impostato e non esiste ancora, lo crea dal template
if HTML_FILE and not os.path.exists(HTML_FILE):
    _template = os.path.join('..', 'network_template.html')
    if os.path.exists(_template):
        import shutil as _shutil_setup
        _shutil_setup.copy(_template, HTML_FILE)
        print(f'Creato {HTML_FILE} dal template')

LAMF_KEYS = ['L1', 'L2', 'L3', 'L4']
LAMF_COLS = [f'lamf_{k}' for k in LAMF_KEYS]

print(f'Materia:   {MATERIA_NAME}')
print(f'Modello:   {LLM_MODEL}')
print(f'Soglia H:  {HYBRIDITY_THRESHOLD}')
print(f'Nodi IT:   {NODES_TEXTS_IT_FILE}  (esiste: {os.path.exists(NODES_TEXTS_IT_FILE)})')
print(f'Diffusion: {NODES_DIFFUSION_FILE}  (esiste: {os.path.exists(NODES_DIFFUSION_FILE)})')
print(f'HTML:      {HTML_FILE or "(disabilitato)"}')

## 2. Caricamento Dati

In [ ]:
# ── Hybridity per atto ─────────────────────────────────────────────────────────
hyb = pd.read_csv(NODES_HYBRIDITY_FILE)
if 'id' in hyb.columns and 'celex' not in hyb.columns:
    hyb = hyb.rename(columns={'id': 'celex'})
print(f'Atti totali: {len(hyb)}')

# ── Per-articolo level scores ──────────────────────────────────────────────────
art_df = pd.read_csv(NODES_LAMFALUSSY_FILE)
print(f'Articoli totali: {len(art_df)}')

# ── Evidence (provisions) per articolo ────────────────────────────────────────
seg_df = pd.read_csv(SEGMENTS_LAMF_FILE)
seg_ok = seg_df[seg_df['llm_status'] == 'ok'].copy()
print(f'Segmenti classificati: {len(seg_ok)}')

# ── Testi completi ─────────────────────────────────────────────────────────────
ART_TEXTS = {}   # (celex, art_id) → testo
FULL_TEXTS = {}  # celex → full_text

for fpath in [NODES_TEXTS_IT_FILE, NODES_TEXTS_EU_FILE]:
    if not fpath or not os.path.exists(fpath):
        continue
    df_txt = pd.read_csv(fpath)
    id_col = next((c for c in ['Id', 'id', 'celex'] if c in df_txt.columns), None)
    if id_col is None:
        print(f'Warning: no id column found in {fpath}, skipping')
        continue
    for _, row in df_txt.iterrows():
        celex = str(row[id_col])
        ft = str(row.get('full_text', '') or '')
        if ft and ft != 'nan':
            FULL_TEXTS[celex] = ft
        segs_raw = row.get('segments', '')
        if not segs_raw or str(segs_raw) in ('nan', '[]', ''):
            continue
        try:
            for seg in json.loads(str(segs_raw)):
                if seg.get('tipo') == 'articolo':
                    idf = str(seg.get('identificatore', ''))
                    ART_TEXTS[(celex, idf)] = str(seg.get('testo', ''))
        except Exception:
            pass

print(f'Testi articoli caricati: {len(ART_TEXTS)}')
print(f'Full texts caricati: {len(FULL_TEXTS)}')

# ── Heatmaps esistenti ────────────────────────────────────────────────────────
with open(HEATMAPS_FILE, encoding='utf-8') as f:
    HEATMAPS = json.load(f)
print(f'Heatmaps: {len(HEATMAPS)} atti')

# ── Archi ─────────────────────────────────────────────────────────────────────
edges_list = []
for ep in [EDGES_IT_FILE, EDGES_EU_IT_FILE]:
    if os.path.exists(ep):
        df_e = pd.read_csv(ep)
        edges_list.append(df_e)
        print(f'Archi da {os.path.basename(ep)}: {len(df_e)}')

# Auto-detect archi focali (da nb02, usati per domini non-appalti come FDI)
_focal_edges_path = os.path.join(output_path, 'edges_focal.csv')
_focal_nodes_path = os.path.join(output_path, 'nodes_focal.csv')
if (os.path.exists(_focal_edges_path) and os.path.exists(_focal_nodes_path)
        and _focal_edges_path not in [EDGES_IT_FILE, EDGES_EU_IT_FILE]):
    _df_fn = pd.read_csv(_focal_nodes_path)
    _id_to_celex = _df_fn.set_index('Id')['Label'].to_dict() if 'Id' in _df_fn.columns and 'Label' in _df_fn.columns else {}
    _df_fe = pd.read_csv(_focal_edges_path)
    if 'Source' in _df_fe.columns and _id_to_celex:
        _df_fe['src_slug'] = _df_fe['Source'].map(_id_to_celex)
        _df_fe['dst_slug'] = _df_fe['Target'].map(_id_to_celex)
        _df_fe['type']     = _df_fe['Type'] if 'Type' in _df_fe.columns else 'CITES'
        _df_fe['family']   = 'ref'
        _df_fe['w']        = 1
        _df_fe = _df_fe[['src_slug', 'dst_slug', 'type', 'family', 'w']].dropna()
        if len(_df_fe) > 0:
            edges_list.append(_df_fe)
            print(f'Archi da edges_focal.csv (con mapping ID→CELEX): {len(_df_fe)}')

edges_df = pd.concat(edges_list, ignore_index=True) if edges_list else pd.DataFrame()
print(f'Archi totali: {len(edges_df)}')

## 3. Fase A — Selezione Candidati e Partizione Deterministica

### Logica di partizione

Per ogni atto ibrido:
1. Ogni articolo viene assegnato al suo livello dominante
2. Si formano blocchi: **L1-block** e **L234-block** (split primario Lamfalussy)
3. Se il L234-block ha abbastanza articoli L4 distinti, si separa ulteriormente in **L23-block** + **L4-block**
4. Blocchi troppo piccoli (< `MIN_BLOCK_SIZE`) vengono assorbiti dal blocco più vicino
5. Si calcola H per ogni blocco risultante — se il miglioramento è < `MIN_H_IMPROVEMENT`, lo split non viene proposto

In [32]:
def normalized_entropy(vals: list[float]) -> float:
    """Entropia normalizzata di una distribuzione (0=pura, 1=massima confusione).
    vals: lista di percentuali (somma ~100 o ~1, entrambe ok).
    """
    if not vals or sum(vals) == 0:
        return 0.0
    ps = [v / sum(vals) for v in vals]
    eps = 1e-9
    H = -sum(p * math.log2(p + eps) for p in ps if p > 0)
    H_max = math.log2(len(vals))
    return round(H / H_max, 4) if H_max > 0 else 0.0


def block_hybridity(articles_subset: pd.DataFrame) -> float:
    """H del blocco = entropia normalizzata della distribuzione aggregata L1-L4."""
    if articles_subset.empty:
        return 0.0
    vals = [float(articles_subset.get(f'lamf_{k}', 0).sum()) for k in LAMF_KEYS]
    total = sum(vals)
    if total == 0:
        return 0.0
    probs = [v / total for v in vals]
    eps = 1e-9
    H = -sum(p * math.log2(p + eps) for p in probs if p > 0)
    H_max = math.log2(len(vals))
    return round(H / H_max, 4) if H_max > 0 else 0.0


def dominant_level(row) -> str:
    vals = {k: float(row.get(f'lamf_{k}', 0) or 0) for k in LAMF_KEYS}
    return max(vals, key=vals.get)


print('Funzioni di entropia definite.')

Funzioni di entropia definite.


In [33]:
# ── Selezione candidati ────────────────────────────────────────────────────────
candidates = hyb[
    (hyb['hybridity_score'] >= HYBRIDITY_THRESHOLD) &
    (hyb['n_articles'] >= MIN_ARTICLES_FOR_SPLIT)
].copy().sort_values('hybridity_score', ascending=False)

print(f'Atti candidati allo split: {len(candidates)} / {len(hyb)}')
print(candidates[['celex', 'hybridity_score', 'dominant_lamf', 'n_articles']].to_string())

Atti candidati allo split: 19 / 32
            celex  hybridity_score dominant_lamf  n_articles
0    dlgs_59_2010         0.742682            L2          86
1       l_90_2024         0.724486            L2          24
2   dlgs_228_2011         0.705018            L2          11
3    dlgs_33_2013         0.687761            L2          53
5      32014L0023         0.665502            L2          55
6      l_241_1990         0.641493            L1          31
7      32014L0024         0.637435            L2          94
8   dlgs_229_2011         0.624611            L2          12
9      l_136_2010         0.616125            L2          16
11     32014L0025         0.611750            L2         110
12   dpr_207_2010         0.609285            L1         334
13   dlgs_36_2023         0.602233            L2         229
14     l_145_2018         0.602077            L2          19
15  dlgs_218_2012         0.573207            L2          10
16   dlgs_97_2016         0.557451            L2  

In [34]:
def generate_partition(celex: str, art_df: pd.DataFrame) -> dict | None:
    """
    Genera la partizione deterministica per un atto.
    Ritorna None se lo split non è giustificato.
    """
    arts = art_df[art_df['celex'] == celex].copy()
    if arts.empty:
        return None

    # Assegna livello dominante a ogni articolo
    arts['dom'] = arts.apply(dominant_level, axis=1)

    H_before = block_hybridity(arts)

    # ── Partizione primaria: L1 vs L2/L3/L4 ──────────────────────────────────
    l1_arts   = arts[arts['dom'] == 'L1']
    l234_arts = arts[arts['dom'].isin(['L2', 'L3', 'L4'])]

    # ── Split secondario: se L4 è sufficientemente rappresentato ─────────────
    l4_arts  = arts[arts['dom'] == 'L4']
    l23_arts = arts[arts['dom'].isin(['L2', 'L3'])]

    use_triple = (
        len(l4_arts)  >= MIN_BLOCK_SIZE and
        len(l23_arts) >= MIN_BLOCK_SIZE
    )

    # ── Costruzione blocchi ───────────────────────────────────────────────────
    if use_triple:
        blocks_raw = [
            ('L1',  l1_arts),
            ('L23', l23_arts),
            ('L4',  l4_arts),
        ]
    else:
        blocks_raw = [
            ('L1',   l1_arts),
            ('L234', l234_arts),
        ]

    # ── Merge blocchi troppo piccoli ──────────────────────────────────────────
    # Se L1 è troppo piccolo, assorbito in L234/L23
    # Se L4 è troppo piccolo (già gestito sopra), assorbito in L23
    final_blocks = []
    absorbed = set()
    for label, block in blocks_raw:
        if label in absorbed:
            continue
        if len(block) < MIN_BLOCK_SIZE:
            # Assorbi nel blocco precedente o nel successivo
            if final_blocks:
                prev_label, prev_arts = final_blocks[-1]
                final_blocks[-1] = (prev_label, pd.concat([prev_arts, block]))
            else:
                # Nessun blocco prima: assorbi nel successivo
                # (gestito al prossimo giro)
                next_blocks = [(l, b) for l2, b in blocks_raw[blocks_raw.index((label, block))+1:] for l in [l2]]
                if next_blocks:
                    absorbed.add(next_blocks[0][0])
                    _, nb = blocks_raw[blocks_raw.index((label, block))+1]
                    final_blocks.append((label, pd.concat([block, nb])))
                else:
                    final_blocks.append((label, block))
            absorbed.add(label)
        else:
            final_blocks.append((label, block))

    # Se rimane un solo blocco, lo split non ha senso
    if len(final_blocks) <= 1:
        return None

    # ── Calcola H post-split (media pesata per n_articoli) ────────────────────
    total_arts = sum(len(b) for _, b in final_blocks)
    H_after = sum(
        block_hybridity(b) * len(b) / total_arts
        for _, b in final_blocks
    )
    H_improvement = (H_before - H_after) / H_before if H_before > 0 else 0

    if H_improvement < MIN_H_IMPROVEMENT:
        return None  # split non abbastanza utile

    # ── Formatta output ───────────────────────────────────────────────────────
    blocks_out = []
    for label, block in final_blocks:
        art_ids = sorted(
            block['articolo_id'].tolist(),
            key=lambda x: int(re.match(r'(\d+)', str(x)).group(1))
                          if re.match(r'\d', str(x)) else 9999
        )
        # Distribuzione aggregata L1-L4 del blocco
        agg_vals = {k: round(float(block[f'lamf_{k}'].mean()), 2) for k in LAMF_KEYS}
        blocks_out.append({
            'block_id':      f"{celex}__{label}",
            'partition_key': label,
            'article_ids':   art_ids,
            'n_articles':    len(block),
            'H_block':       block_hybridity(block),
            'lamf_avg':      agg_vals,
            'proposed_title': None,  # compilato dal LLM
        })

    return {
        'celex':         celex,
        'H_before':      round(H_before, 4),
        'H_after':       round(H_after, 4),
        'H_improvement': round(H_improvement, 4),
        'n_blocks':      len(final_blocks),
        'blocks':        blocks_out,
        'validated':     False,  # flag: True dopo validazione LLM
    }


# ── Esegui su tutti i candidati ───────────────────────────────────────────────
split_proposals = {}
skipped = []

for _, row in candidates.iterrows():
    celex = str(row['celex'])
    result = generate_partition(celex, art_df)
    if result is not None:
        split_proposals[celex] = result
    else:
        skipped.append(celex)

print(f'Proposte generate:  {len(split_proposals)}')
print(f'Candidati scartati: {len(skipped)} (H improvement < soglia o blocchi troppo piccoli)')
for celex, prop in split_proposals.items():
    print(f"  {celex:<25}  H: {prop['H_before']:.3f} → {prop['H_after']:.3f}  "
          f"(Δ {prop['H_improvement']:.1%})  {prop['n_blocks']} blocchi")

Proposte generate:  15
Candidati scartati: 4 (H improvement < soglia o blocchi troppo piccoli)
  dlgs_59_2010               H: 0.743 → 0.327  (Δ 56.0%)  3 blocchi
  l_90_2024                  H: 0.725 → 0.445  (Δ 38.5%)  2 blocchi
  dlgs_228_2011              H: 0.705 → 0.472  (Δ 33.1%)  2 blocchi
  dlgs_33_2013               H: 0.688 → 0.390  (Δ 43.3%)  3 blocchi
  32014L0023                 H: 0.665 → 0.443  (Δ 33.5%)  2 blocchi
  32014L0024                 H: 0.637 → 0.335  (Δ 47.5%)  3 blocchi
  l_136_2010                 H: 0.616 → 0.292  (Δ 52.5%)  2 blocchi
  32014L0025                 H: 0.612 → 0.332  (Δ 45.8%)  3 blocchi
  dpr_207_2010               H: 0.609 → 0.164  (Δ 73.1%)  3 blocchi
  dlgs_36_2023               H: 0.602 → 0.348  (Δ 42.2%)  3 blocchi
  l_145_2018                 H: 0.602 → 0.315  (Δ 47.7%)  2 blocchi
  dlgs_97_2016               H: 0.557 → 0.280  (Δ 49.8%)  2 blocchi
  dl_13_2023                 H: 0.551 → 0.362  (Δ 34.4%)  2 blocchi
  dlgs_209_2024      

## 4. Fase B — Validazione LLM

Per ogni proposta, il modello riceve:
- Il titolo dell'atto originale
- Per ogni blocco: lista degli articoli con **testo completo** e livello dominante

Il modello può:
1. **Confermare** la partizione
2. **Spostare** articoli borderline tra blocchi (solo se il dominant era < 55%)
3. **Proporre un titolo** per ogni blocco

In [35]:
VALIDATION_SYSTEM = """You are an expert in European legislative techniques and the Lamfalussy framework.
You analyze proposals for separating hybrid legislative acts.
Respond ONLY with valid JSON, no additional text."""


def build_validation_prompt(celex: str, prop: dict, art_df: pd.DataFrame) -> str:
    title_atto = art_df[art_df['celex'] == celex]['title_atto'].iloc[0] \
                 if 'title_atto' in art_df.columns and not art_df[art_df['celex'] == celex].empty \
                 else celex

    blocks_desc = []
    for b in prop['blocks']:
        art_entries = []
        for art_id in b['article_ids']:
            testo = ART_TEXTS.get((celex, str(art_id)), '')
            # Livello dominante dell'articolo
            art_row = art_df[
                (art_df['celex'] == celex) &
                (art_df['articolo_id'].astype(str) == str(art_id))
            ]
            dom = art_row['dominant_lamf'].iloc[0] if not art_row.empty else '?'
            lamf_vals = {k: round(float(art_row[f'lamf_{k}'].iloc[0]), 1)
                         for k in LAMF_KEYS} if not art_row.empty else {}
            art_entries.append(
                f"  Articolo {art_id} [dominante: {dom}, scores: {lamf_vals}]\n"
                f"  {testo}"  # testo completo
            )
        block_str = (
            f"BLOCCO {b['partition_key']} ({b['n_articles']} articoli, "
            f"H_block={b['H_block']:.3f}):\n" +
            "\n---\n".join(art_entries)
        )
        blocks_desc.append(block_str)

    blocks_text = "\n\n".join(blocks_desc)

    return f"""Original Act: "{title_atto}" (celex: {celex})
Original hybridity score: {prop['H_before']:.3f}

The following separation has been proposed in {prop['n_blocks']} distinct acts,
based on the dominant Lamfalussy levels of each article:

{blocks_text}

INSTRUCTIONS:
1. Evaluate whether the separation makes sense from a regulatory content perspective.
2. If an article seems clearly out of place in its block (and its dominance level was less than 55%), 
suggest which block it belongs to.
Do not move articles with a clear dominance (≥ 55%) simply because of thematic preference.
3. Propose a short title (max 10 words) for each resulting block.
4. Each block must contain at least 2 articles—do not propose moves
that would empty a block.

Please reply with this exact JSON:
{{
  "validation": "confirmed" | "adjusted",
  "blocks": [
    {{
      "partition_key": "<stesso key del blocco>",
      "article_ids": ["<lista definitiva articoli>"],
      "proposed_title": "<titolo breve>",
      "rationale": "<max 20 parole>"
    }}
  ],
  "moves": [
    {{ "article_id": "<id>", "from_block": "<key>", "to_block": "<key>", "reason": "<max 15 parole>" }}
  ]
}}"""


print('Prompt di validazione definito.')

Prompt di validazione definito.


In [ ]:
# ── Carica checkpoint se esiste ────────────────────────────────────────────────
validated_splits = {}
if os.path.exists(SPLITS_CKPT_FILE):
    with open(SPLITS_CKPT_FILE, encoding='utf-8') as f:
        ckpt = json.load(f)
    validated_splits = ckpt.get('validated', {})
    print(f'Checkpoint: {len(validated_splits)} atti già validati')


def call_validation_llm(celex: str, prop: dict) -> dict | None:
    """Chiama il LLM per validare la proposta. Ritorna il JSON parsato o None."""
    prompt = build_validation_prompt(celex, prop, art_df)
    for attempt in range(LLM_MAX_RETRIES):
        try:
            resp = client.chat.completions.create(
                model=LLM_MODEL,
                max_tokens=LLM_MAX_TOKENS,
                temperature=0.0,
                messages=[
                    {"role": "system", "content": VALIDATION_SYSTEM},
                    {"role": "user",   "content": prompt},
                ]
            )
            raw = resp.choices[0].message.content.strip()
            raw = re.sub(r'^```json|^```|```$', '', raw, flags=re.MULTILINE).strip()
            return json.loads(raw)
        except (json.JSONDecodeError, openai.RateLimitError) as e:
            print(f'    [{celex}] tentativo {attempt+1}/{LLM_MAX_RETRIES}: {e}')
            time.sleep(LLM_RETRY_DELAY)
        except Exception as e:
            print(f'    [{celex}] errore: {e}')
            return None
    return None


def apply_llm_validation(celex: str, prop: dict, llm_result: dict) -> dict:
    """
    Applica il risultato LLM alla proposta.
    Aggiorna article_ids di ogni blocco e proposed_title.
    """
    prop = prop.copy()
    prop['blocks'] = [b.copy() for b in prop['blocks']]

    # Mappa key → blocco
    block_map = {b['partition_key']: b for b in prop['blocks']}

    # Applica spostamenti suggeriti
    if llm_result.get('validation') == 'adjusted':
        for move in llm_result.get('moves', []):
            art_id   = str(move.get('article_id', ''))
            from_key = move.get('from_block', '')
            to_key   = move.get('to_block', '')
            if from_key in block_map and to_key in block_map:
                src = block_map[from_key]
                dst = block_map[to_key]
                if art_id in src['article_ids'] and (len(src['article_ids']) - 1) >= MIN_BLOCK_SIZE:
                    src['article_ids'].remove(art_id)
                    src['n_articles'] -= 1
                    dst['article_ids'].append(art_id)
                    dst['n_articles'] += 1

    # Applica titoli proposti
    for llm_block in llm_result.get('blocks', []):
        key = llm_block.get('partition_key', '')
        if key in block_map:
            block_map[key]['proposed_title'] = llm_block.get('proposed_title', '')
            block_map[key]['llm_rationale']  = llm_block.get('rationale', '')
            if llm_block.get('article_ids'):
                block_map[key]['article_ids'] = [str(a) for a in llm_block['article_ids']]
                block_map[key]['n_articles']  = len(block_map[key]['article_ids'])

    prop['validated']      = True
    prop['llm_validation'] = llm_result.get('validation', 'unknown')
    return prop


# ── Esegui validazione con checkpoint ─────────────────────────────────────────
lock = Lock()
to_validate = {k: v for k, v in split_proposals.items() if k not in validated_splits}
print(f'Da validare: {len(to_validate)} atti')


def validate_one(item):
    celex, prop = item
    time.sleep(LLM_DELAY_SECONDS)
    llm_result = call_validation_llm(celex, prop)
    if llm_result is None:
        prop['validated']      = True
        prop['llm_validation'] = 'fallback'
        for b in prop['blocks']:
            b['proposed_title'] = f"Parte {b['partition_key']}"
        return celex, prop
    return celex, apply_llm_validation(celex, prop, llm_result)


with ThreadPoolExecutor(max_workers=MAX_WORKERS) as ex:
    futures = {ex.submit(validate_one, item): item[0] for item in to_validate.items()}
    for i, fut in enumerate(as_completed(futures), 1):
        celex, validated_prop = fut.result()
        with lock:
            validated_splits[celex] = validated_prop
            if i % CHECKPOINT_EVERY == 0:
                with open(SPLITS_CKPT_FILE, 'w', encoding='utf-8') as f:
                    json.dump({'validated': validated_splits}, f, ensure_ascii=False)
                print(f'  Checkpoint: {i}/{len(to_validate)} validati')
        print(f'  [{i}/{len(to_validate)}] {celex}: {validated_prop["llm_validation"]}')

# Checkpoint finale
with open(SPLITS_CKPT_FILE, 'w', encoding='utf-8') as f:
    json.dump({'validated': validated_splits}, f, ensure_ascii=False)
print(f'\nValidazione completa: {len(validated_splits)} atti')

# ── Ricalcola H_block e lamf_avg post-LLM ─────────────────────────────────────
# apply_llm_validation può spostare articoli tra blocchi senza aggiornare
# H_block/lamf_avg; lo stesso vale per splits caricati da checkpoint.
for celex, prop in validated_splits.items():
    total_arts = sum(b['n_articles'] for b in prop['blocks'])
    H_after_recalc = 0.0
    for b in prop['blocks']:
        arts_subset = art_df[
            (art_df['celex'] == celex) &
            (art_df['articolo_id'].astype(str).isin([str(a) for a in b['article_ids']]))
        ]
        b['H_block'] = block_hybridity(arts_subset)
        if not arts_subset.empty:
            b['lamf_avg'] = {k: round(float(arts_subset[f'lamf_{k}'].mean()), 2)
                             for k in LAMF_KEYS}
        H_after_recalc += b['H_block'] * b['n_articles'] / total_arts if total_arts > 0 else 0
    prop['H_after']       = round(H_after_recalc, 4)
    prop['H_improvement'] = round((prop['H_before'] - prop['H_after']) / prop['H_before'], 4) \
                            if prop['H_before'] > 0 else 0.0

print('\nH_block e lamf_avg ricalcolati post-LLM:')
for celex, prop in validated_splits.items():
    print(f"  {celex:<25} H: {prop['H_before']:.3f} → {prop['H_after']:.3f}  (Δ {prop['H_improvement']:.1%})")

## 5. Fase C — Attribuzione Archi a Livello Articolo

Ogni arco `(A → B)` nella rete attuale è a livello di documento.
Per redistribuirlo correttamente dopo lo split, bisogna sapere **quale articolo di A** contiene la citazione a B.

Questa fase scansiona i testi articolo per articolo e costruisce un indice:
`article_edge_index[(src_celex, dst_celex)] = [art_id1, art_id2, ...]`

In [37]:
# ── Pattern di citazione (stesso di notebook 04) ───────────────────────────────
TIPO_ALIASES = {
    'decreto legislativo': 'dlgs', 'decreto-legislativo': 'dlgs',
    'd.lgs': 'dlgs', 'd.lgs.': 'dlgs', 'dlgs': 'dlgs',
    'legge': 'l', 'l.': 'l',
    'decreto del presidente della repubblica': 'dpr',
    'd.p.r': 'dpr', 'd.p.r.': 'dpr', 'dpr': 'dpr',
    'decreto-legge': 'dl', 'decreto legge': 'dl', 'd.l.': 'dl', 'dl': 'dl',
    'decreto ministeriale': 'dm', 'decreto del ministro': 'dm',
    'd.m.': 'dm', 'dm': 'dm',
}

MESI = r'(?:gennaio|febbraio|marzo|aprile|maggio|giugno|luglio|agosto|settembre|ottobre|novembre|dicembre)'
PAT_FULL = re.compile(
    r'(decreto(?:\s+del\s+presidente\s+della\s+repubblica|[\s\-]?legislativo|[\s\-]?legge|[\s]ministeriale)?'
    r'|legge|l\.)\s+'
    r'(?:\d{1,2}\s+' + MESI + r'\s+)?'
    r'(\d{4})\s*,?\s*n\.?\s*(\d+)',
    re.IGNORECASE
)
PAT_SHORT = re.compile(
    r'(d\.lgs\.|d\.p\.r\.|d\.l\.|d\.m\.|legge)\s*'
    r'(?:\d{1,2}\s+' + MESI + r'\s+)?'
    r'(\d{4})\s*[,\s]+n\.?\s*(\d+)',
    re.IGNORECASE
)


def normalize_tipo(raw: str) -> str | None:
    raw = raw.strip().lower().rstrip('.')
    for alias, norm in TIPO_ALIASES.items():
        if raw.startswith(alias):
            return norm
    return None


# Costruisce lookup slug → (tipo, anno, numero)
all_nodes_df = pd.read_csv(NODES_TEXTS_IT_FILE)
id_col = 'Id' if 'Id' in all_nodes_df.columns else 'id'
slug_lookup = {}
for _, row in all_nodes_df.iterrows():
    slug = str(row[id_col])
    parts = slug.split('_')
    if len(parts) >= 3:
        slug_lookup[(parts[0], parts[2], parts[1])] = slug

print(f'Slug lookup: {len(slug_lookup)} voci')

# ── Scansione articolo per articolo ───────────────────────────────────────────
# Interessa solo gli atti coinvolti negli split (sorgente o destinazione)
split_celexes = set(validated_splits.keys())
all_edge_celexes = set(edges_df['src_slug'].tolist() + edges_df['dst_slug'].tolist()) \
                   if len(edges_df) else set()
relevant_celexes = split_celexes  # sorgenti: solo atti splittati

# article_edge_index[(src_celex, dst_celex)] = set di art_id sorgente
article_edge_index = {}

for celex in relevant_celexes:
    arts = art_df[art_df['celex'] == celex]
    for _, art_row in arts.iterrows():
        art_id = str(art_row['articolo_id'])
        testo = ART_TEXTS.get((celex, art_id), '')
        if not testo:
            continue
        for pat in [PAT_FULL, PAT_SHORT]:
            for m in pat.finditer(testo):
                tipo_raw = m.group(1)
                anno     = m.group(2)
                numero   = m.group(3)
                tipo_n   = normalize_tipo(tipo_raw)
                if not tipo_n:
                    continue
                dst_celex = slug_lookup.get((tipo_n, anno, numero))
                if not dst_celex or dst_celex == celex:
                    continue
                key = (celex, dst_celex)
                article_edge_index.setdefault(key, set()).add(art_id)

print(f'Coppie (src_atto, dst_atto) con attribuzione articolo: {len(article_edge_index)}')

Slug lookup: 29 voci
Coppie (src_atto, dst_atto) con attribuzione articolo: 58


## 6. Fase D — Risoluzione Edge con Target Ambiguo (LLM)

Un edge `A1 → B` è ambiguo quando B è stato splittato in B1 e B2,
e dal testo dell'articolo sorgente non è chiaro a quale parte di B ci si riferisce.

Il LLM legge il testo dell'articolo sorgente + i titoli/descrizioni di B1 e B2
e decide il target corretto (o entrambi, se la citazione è davvero generica).

In [38]:
EDGE_RESOLVE_SYSTEM = """Sei un esperto di diritto europeo e tecnica legislativa.
Analizza citazioni normative e stabilisci a quale parte di una legge splittata si riferiscono.
Rispondi SOLO con JSON valido."""


def build_edge_resolve_prompt(
    src_celex: str, art_id: str,
    dst_celex: str, dst_blocks: list
) -> str:
    art_text = ART_TEXTS.get((src_celex, str(art_id)), '')

    blocks_desc = []
    for b in dst_blocks:
        art_summaries = []
        for a_id in b['article_ids'][:5]:  # prime 5 per brevità
            t = ART_TEXTS.get((dst_celex, str(a_id)), '')[:300]
            art_summaries.append(f"  Art.{a_id}: {t}...")
        blocks_desc.append(
            f"PARTE {b['partition_key']} — \"{b.get('proposed_title', b['partition_key'])}\""
            f" ({b['n_articles']} articoli, livello dominante: {b['partition_key']})\n"
            + "\n".join(art_summaries)
        )

    return f"""L'articolo seguente cita la legge "{dst_celex}", che è stata divisa in {len(dst_blocks)} parti:

TESTO ARTICOLO SORGENTE (celex: {src_celex}, art: {art_id}):
{art_text}

LE DUE PARTI DELLA LEGGE CITATA:
{'\n\n'.join(blocks_desc)}

DOMANDA: La citazione nell'articolo sorgente si riferisce a:
- Solo alla Parte {dst_blocks[0]['partition_key']}
- Solo alla Parte {dst_blocks[-1]['partition_key']}
- A entrambe le parti

Rispondi con:
{{"target": ["{dst_blocks[0]['partition_key']}"] | ["{dst_blocks[-1]['partition_key']}"] | ["{dst_blocks[0]['partition_key']}", "{dst_blocks[-1]['partition_key']}"],
  "reason": "<max 15 parole>"}}"""


def resolve_edge_target(src_celex, art_id, dst_celex, dst_blocks) -> list[str]:
    """Ritorna lista di partition_key del target (uno o entrambi)."""
    prompt = build_edge_resolve_prompt(src_celex, art_id, dst_celex, dst_blocks)
    for attempt in range(LLM_MAX_RETRIES):
        try:
            resp = client.chat.completions.create(
                model=LLM_MODEL,
                max_tokens=200,
                temperature=0.0,
                messages=[
                    {"role": "system", "content": EDGE_RESOLVE_SYSTEM},
                    {"role": "user",   "content": prompt},
                ]
            )
            raw = resp.choices[0].message.content.strip()
            raw = re.sub(r'^```json|^```|```$', '', raw, flags=re.MULTILINE).strip()
            parsed = json.loads(raw)
            return parsed.get('target', [dst_blocks[0]['partition_key']])
        except Exception as e:
            print(f'    edge resolve error (attempt {attempt+1}): {e}')
            time.sleep(LLM_RETRY_DELAY)
    # Fallback: target = blocco con più citazioni tematiche (primo blocco)
    return [dst_blocks[0]['partition_key']]


print('Funzioni di risoluzione edge definite.')

Funzioni di risoluzione edge definite.


## 7. Fase E — Redistribuzione Archi

In [39]:
# ── Mappa: block_id → set di article_ids ──────────────────────────────────────
# block_id = f"{celex}__{partition_key}"
block_art_map = {}  # block_id → frozenset(article_ids)
for celex, prop in validated_splits.items():
    for b in prop['blocks']:
        block_art_map[b['block_id']] = frozenset(str(a) for a in b['article_ids'])


def art_to_block(celex: str, art_id: str) -> str | None:
    """Dato celex + article_id, ritorna il block_id a cui appartiene dopo lo split."""
    if celex not in validated_splits:
        return celex  # non splittato: blocco = atto stesso
    for b in validated_splits[celex]['blocks']:
        if str(art_id) in [str(a) for a in b['article_ids']]:
            return b['block_id']
    return celex  # fallback


# ── Genera nuovi archi ─────────────────────────────────────────────────────────
new_edges = []   # {source, target, type, family, w, resolved_by}
ambiguous_resolved = 0
edge_errors = 0

if len(edges_df) == 0:
    print('Nessun arco da redistribuire.')
else:
    for _, edge_row in edges_df.iterrows():
        src = str(edge_row.get('src_slug', edge_row.get('source', '')))
        dst = str(edge_row.get('dst_slug', edge_row.get('target', '')))
        etype  = str(edge_row.get('type',   'CITES'))
        family = str(edge_row.get('family', 'ref'))
        w_raw  = edge_row.get('w', 1)
        try:
            w = int(float(w_raw))
        except Exception:
            w = 1

        src_is_split = src in validated_splits
        dst_is_split = dst in validated_splits

        # Caso 1: né src né dst splittati → arco invariato
        if not src_is_split and not dst_is_split:
            new_edges.append({'source': src, 'target': dst, 'type': etype,
                              'family': family, 'w': w, 'resolved_by': 'unchanged'})
            continue

        # Trova quali articoli di src citano dst
        citing_arts = list(article_edge_index.get((src, dst), set()))

        if not citing_arts:
            # Non abbiamo info articolo-livello: mantieni arco a livello atto
            new_src = src if not src_is_split else validated_splits[src]['blocks'][0]['block_id']
            new_dst = dst if not dst_is_split else validated_splits[dst]['blocks'][0]['block_id']
            new_edges.append({'source': new_src, 'target': new_dst, 'type': etype,
                              'family': family, 'w': w, 'resolved_by': 'no_art_info'})
            edge_errors += 1
            continue

        # Mappa gli articoli sorgente ai loro blocchi
        src_blocks_used = set()
        for art_id in citing_arts:
            block = art_to_block(src, art_id)
            src_blocks_used.add(block)

        # Per ogni blocco sorgente, determina il target
        for src_block in src_blocks_used:
            if not dst_is_split:
                new_edges.append({'source': src_block, 'target': dst, 'type': etype,
                                  'family': family, 'w': w, 'resolved_by': 'article_match'})
            else:
                # Caso 2: dst è splittato → LLM decide quale blocco
                # Prende gli articoli citanti appartenenti a questo src_block
                arts_in_block = [
                    a for a in citing_arts
                    if art_to_block(src, a) == src_block
                ]

                # Chiamata LLM per ogni articolo sorgente distinto
                dst_blocks = validated_splits[dst]['blocks']
                target_keys = set()
                for art_id in arts_in_block:
                    keys = resolve_edge_target(src, art_id, dst, dst_blocks)
                    target_keys.update(keys)
                    ambiguous_resolved += 1
                    time.sleep(LLM_DELAY_SECONDS)

                for tkey in target_keys:
                    dst_block_id = f"{dst}__{tkey}"
                    new_edges.append({'source': src_block, 'target': dst_block_id,
                                      'type': etype, 'family': family, 'w': w,
                                      'resolved_by': 'llm_target'})

    # Deduplica (stessa coppia source-target-type)
    seen = set()
    dedup_edges = []
    for e in new_edges:
        key = (e['source'], e['target'], e['type'])
        if key not in seen:
            seen.add(key)
            dedup_edges.append(e)
    new_edges = dedup_edges

    print(f'Archi nuovi generati: {len(new_edges)}')
    print(f'  - invariati:          {sum(1 for e in new_edges if e["resolved_by"]=="unchanged")}')
    print(f'  - per articolo:       {sum(1 for e in new_edges if e["resolved_by"]=="article_match")}')
    print(f'  - risolti da LLM:     {sum(1 for e in new_edges if e["resolved_by"]=="llm_target")}')
    print(f'  - senza info art.:    {edge_errors}')

Archi nuovi generati: 132
  - invariati:          11
  - per articolo:       53
  - risolti da LLM:     48
  - senza info art.:    20


## 8. Fase F — Calcolo H Globale Prima e Dopo

In [40]:
# H globale PRIMA = media degli hybridity_score originali su tutti gli atti
H_global_before = round(float(hyb['hybridity_score'].mean()), 4)

# H globale DOPO:
# - Atti non splittati: mantengono il loro hybridity_score originale
# - Atti splittati: sostituiti dai loro blocchi, ognuno con H_block
h_scores_after = []
for _, row in hyb.iterrows():
    celex = str(row['celex'])
    if celex in validated_splits:
        prop = validated_splits[celex]
        # Ogni blocco è un nuovo "atto" nel dopo
        for b in prop['blocks']:
            h_scores_after.append(b['H_block'])
    else:
        h_scores_after.append(float(row['hybridity_score']))

H_global_after = round(float(np.mean(h_scores_after)), 4)
H_improvement  = round((H_global_before - H_global_after) / H_global_before, 4)

n_atti_before = len(hyb)
n_atti_after  = len(h_scores_after)

print(f'H globale PRIMA:  {H_global_before:.4f}  ({n_atti_before} atti)')
print(f'H globale DOPO:   {H_global_after:.4f}  ({n_atti_after} atti)')
print(f'Miglioramento:    {H_improvement:.1%}')
print(f'Atti splittati:   {len(validated_splits)} → {sum(p["n_blocks"] for p in validated_splits.values())} blocchi')

H globale PRIMA:  0.4252  (32 atti)
H globale DOPO:   0.3031  (53 atti)
Miglioramento:    28.7%
Atti splittati:   15 → 36 blocchi


## 9. Fase G — Entropia Globale via Diffusione di Citazioni

L'entropia locale `H_block` misura solo l'ibridità interna di ogni blocco.
`H_global` cattura anche la **contaminazione funzionale** trasmessa dagli archi:
un blocco coeso che cita atti ibridi eredita complessità nel sistema in cui opera.

Stesso metodo di notebook 04 Fase E, applicato alla rete post-split:

$$
\mathbf{P}_{\text{glob}} = (1-\lambda)(I - \lambda \tilde{W})^{-1} \mathbf{P}_{\text{loc}}
$$

Gli archi `recep` (EU→IT) vengono invertiti: l'atto italiano eredita complessità dalla direttiva che recepisce.

In [ ]:
LAMBDA = 0.5

# ── 1. Nodi rete post-split ───────────────────────────────────────────────────
post_nodes = []

for _, row in hyb.iterrows():
    celex = str(row['celex'])
    if celex in validated_splits:
        continue
    vals = np.array([float(row.get(col, 0) or 0) for col in LAMF_COLS])
    s = vals.sum()
    post_nodes.append((celex, vals / s if s > 0 else vals))

for celex, prop in validated_splits.items():
    for b in prop['blocks']:
        vals = np.array([float(b.get('lamf_avg', {}).get(k, 0) or 0) for k in LAMF_KEYS])
        s = vals.sum()
        post_nodes.append((b['block_id'], vals / s if s > 0 else vals))

N   = len(post_nodes)
idx = {nid: i for i, (nid, _) in enumerate(post_nodes)}

P_loc = np.zeros((N, 4))
for i, (_, p) in enumerate(post_nodes):
    P_loc[i] = p

# ── 2. Matrice W dagli archi post-split ───────────────────────────────────────
W = np.zeros((N, N))
for e in new_edges:
    s_id, t_id = e.get('source', ''), e.get('target', '')
    fam, w = e.get('family', 'ref'), float(e.get('w', 1))
    if s_id not in idx or t_id not in idx:
        continue
    if fam == 'recep':
        s_id, t_id = t_id, s_id
    W[idx[s_id], idx[t_id]] += w

row_sums = W.sum(axis=1, keepdims=True)
row_sums[row_sums == 0] = 1.0
W_tilde = W / row_sums

# ── 3. Diffusione ─────────────────────────────────────────────────────────────
P_glob = (1 - LAMBDA) * np.linalg.solve(np.eye(N) - LAMBDA * W_tilde, P_loc)
ps = P_glob.sum(axis=1, keepdims=True)
ps[ps == 0] = 1.0
P_glob = P_glob / ps

# ── 4. H_global e lamf_glob per nodo ─────────────────────────────────────────
def shannon_norm4(p, eps=1e-9):
    p = np.clip(p, 0, 1) / (np.clip(p, 0, 1).sum() + eps)
    return float(np.clip(-np.sum(p * np.log2(p + eps)) / np.log2(4), 0, 1))

H_global_post = {nid: shannon_norm4(P_glob[i]) for i, (nid, _) in enumerate(post_nodes)}

lamf_glob_post = {
    nid: {k: round(float(P_glob[i, j] * 100), 1) for j, k in enumerate(LAMF_KEYS)}
    for i, (nid, _) in enumerate(post_nodes)
}

# ── 5. H_global PRIMA ─────────────────────────────────────────────────────────
if HTML_FILE and os.path.exists(HTML_FILE):
    # Legge dall'HTML (appalti_it: nb04 Fase E ha già scritto H_global sui nodi)
    with open(HTML_FILE, 'r', encoding='utf-8') as _f:
        _html_g = _f.read()
    _nodes_html = json.loads(re.search(r'const NODES\s*=\s*(\[.*?\]);', _html_g, re.DOTALL).group(1))
    H_global_before_diff = round(float(np.mean([n.get('H_global', n.get('H', 0.0)) for n in _nodes_html])), 4)
    H_local_before_diff  = round(float(np.mean([n.get('H_local',  n.get('H', 0.0)) for n in _nodes_html])), 4)
    print(f'H_global caricato dall\'HTML: {H_global_before_diff:.4f}')
elif os.path.exists(NODES_DIFFUSION_FILE):
    # Legge da nodes_diffusion.csv (prodotto da nb04 Fase E standalone)
    _diff = pd.read_csv(NODES_DIFFUSION_FILE)
    H_global_before_diff = round(float(_diff['H_global'].mean()), 4)
    H_local_before_diff  = round(float(_diff['H_local'].mean()),  4)
    print(f'H_global caricato da nodes_diffusion.csv: mean={H_global_before_diff:.4f}  (vs H_local={H_local_before_diff:.4f})')
else:
    # Fallback proxy
    H_global_before_diff = H_global_before
    H_local_before_diff  = H_global_before
    print('nodes_diffusion.csv non trovato — H_global = H_local (proxy)')

# ── 6. H_global DOPO ──────────────────────────────────────────────────────────
H_local_after_arr  = np.array([shannon_norm4(P_loc[i])  for i in range(N)])
H_global_after_arr = np.array([shannon_norm4(P_glob[i]) for i in range(N)])

H_local_after_diff  = round(float(H_local_after_arr.mean()), 4)
H_global_after_diff = round(float(H_global_after_arr.mean()), 4)

H_global_improvement_diff = round((H_global_before_diff - H_global_after_diff) / H_global_before_diff, 4) \
                             if H_global_before_diff > 0 else 0.0

# ── 7. Riepilogo ──────────────────────────────────────────────────────────────
print(f'\nλ = {LAMBDA}  |  Nodi dopo split: {N}')
print()
print(f'{"":30}  {"PRIMA":>8}  {"DOPO":>8}  {"Δ":>7}')
print(f'{"H_local  (media atti)":30}  {H_local_before_diff:8.4f}  {H_local_after_diff:8.4f}  '
      f'{H_local_after_diff - H_local_before_diff:+7.4f}')
print(f'{"H_global (media atti)":30}  {H_global_before_diff:8.4f}  {H_global_after_diff:8.4f}  '
      f'{H_global_after_diff - H_global_before_diff:+7.4f}')
print(f'{"  (riduzione H_global)":30}  {"":8}  {"":8}  {H_global_improvement_diff:+7.1%}')
print()
print(f'{"block_id":<32} {"H_local":>8} {"H_global":>9} {"Δ":>7}')
for celex, prop in validated_splits.items():
    for b in prop['blocks']:
        bid = b['block_id']
        hl  = shannon_norm4(P_loc[idx[bid]])
        hg  = H_global_post[bid]
        print(f'{bid:<32} {hl:8.3f} {hg:9.3f} {hg-hl:+7.3f}')

In [ ]:
def build_block_heatmap(celex: str, article_ids: list) -> list:
    """
    Costruisce i dati heatmap per un blocco a partire dal HEATMAPS esistente.
    Filtra solo gli articoli del blocco, mantiene la struttura originale.
    """
    orig_heatmap = HEATMAPS.get(celex, [])
    art_id_set   = {str(a) for a in article_ids}
    return [a for a in orig_heatmap if str(a.get('id', '')) in art_id_set]


# ── Costruisce il JSON finale ──────────────────────────────────────────────────
splits_out = {
    'meta': {
        # Entropia locale (media non pesata degli atti/blocchi)
        'H_local_before':      H_global_before,
        'H_local_after':       H_global_after,
        'H_local_improvement': H_improvement,
        # Entropia globale via diffusione (stessa formula di notebook 04 Fase E)
        'H_global_before':     H_global_before_diff,
        'H_global_after':      H_global_after_diff,
        'H_global_improvement': H_global_improvement_diff,
        'lambda':              LAMBDA,
        # Conteggi
        'n_acts_before': n_atti_before,
        'n_acts_after':  n_atti_after,
        'n_split':       len(validated_splits),
        'model':         LLM_MODEL,
        'thresholds': {
            'hybridity':         HYBRIDITY_THRESHOLD,
            'min_articles':      MIN_ARTICLES_FOR_SPLIT,
            'min_block':         MIN_BLOCK_SIZE,
            'min_h_improvement': MIN_H_IMPROVEMENT,
        }
    },
    'splits': {},
    'edges_after': new_edges if new_edges else [],
}

for celex, prop in validated_splits.items():
    blocks_out = []
    for b in prop['blocks']:
        blocks_out.append({
            'block_id':       b['block_id'],
            'partition_key':  b['partition_key'],
            'proposed_title': b.get('proposed_title') or b['partition_key'],
            'llm_rationale':  b.get('llm_rationale', ''),
            'article_ids':    [str(a) for a in b['article_ids']],
            'n_articles':     b['n_articles'],
            'H_block':        b['H_block'],
            'H_global':       round(H_global_post.get(b['block_id'], 0.0), 4),
            'lamf_avg':       b.get('lamf_avg', {}),
            'lamf_glob':      lamf_glob_post.get(b['block_id'], {}),
            'heatmap':        build_block_heatmap(celex, b['article_ids']),
        })

    splits_out['splits'][celex] = {
        'celex':          celex,
        'H_before':       prop['H_before'],
        'H_after':        prop['H_after'],
        'H_improvement':  prop['H_improvement'],
        'n_blocks':       prop['n_blocks'],
        'llm_validation': prop.get('llm_validation', 'unknown'),
        'blocks':         blocks_out,
    }

# ── Salva nella cartella del dominio ──────────────────────────────────────────
with open(SPLITS_JSON_FILE, 'w', encoding='utf-8') as f:
    json.dump(splits_out, f, ensure_ascii=False)

size_kb = os.path.getsize(SPLITS_JSON_FILE) // 1024
print(f'✓ Salvato: {SPLITS_JSON_FILE}  ({size_kb} KB)')
print(f'  Atti splittati: {len(splits_out["splits"])}')
print(f'  Archi nel grafo dopo: {len(splits_out["edges_after"])}')
print()
print(f'  H_local  PRIMA→DOPO: {H_global_before:.4f} → {H_global_after:.4f}  ({H_improvement:+.1%})')
print(f'  H_global PRIMA→DOPO: {H_global_before_diff:.4f} → {H_global_after_diff:.4f}  ({H_global_improvement_diff:+.1%})')
print()
print('Riepilogo blocchi:')
for celex, s in splits_out['splits'].items():
    print(f"  {celex:<25} H_loc {s['H_before']:.3f}→{s['H_after']:.3f}  "
          f"({s['n_blocks']} blocchi): "
          + ', '.join(f'"{b["proposed_title"]}"' for b in s['blocks']))

# Se HTML abilitato, copia splits.json accanto all'HTML per la visualizzazione
if HTML_FILE:
    import shutil as _shutil_sp
    _html_dir_s  = os.path.dirname(os.path.abspath(HTML_FILE))
    _html_splits = os.path.join(_html_dir_s, 'splits.json')
    if os.path.abspath(SPLITS_JSON_FILE) != _html_splits:
        _shutil_sp.copy(SPLITS_JSON_FILE, _html_splits)
        print(f'\n✓ Copiato anche in {_html_splits}')